<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 8 · ¿Quiénes son mis clientes y cuánto valen?

Corte de medio semestre. Las siete semanas anteriores se juntan aquí: cargas datos, los limpias, los
agrupas, los unes, los dibujas y los cuentas. La diferencia es que hoy el resultado no es una tabla ni
una figura, sino **una lista de clientes con nombre de segmento y una frase de qué se le dice a cada
uno el lunes**. Empiezas resolviendo las dos cosas que arruinan cualquier análisis de clientes —las
devoluciones y las ventas que nadie sabe a quién atribuir—, construyes recencia, frecuencia, monto y
margen, los conviertes en segmentos con nombre comercial, y al final dejas que un algoritmo de
agrupamiento haga el mismo trabajo para ver en qué te gana y en qué no. Sin trampa: la comparación
final es honesta y una de las dos pierde.

> **Hoy haces** · Resuelves devoluciones y clientes sin identificar con dos decisiones escritas y
> cuantificadas (90 min y el trabajo del receso). Calculas R, F, M y P por cliente, los conviertes en
> quintiles con `qcut`, construyes la matriz RFM y **nombras seis segmentos con lenguaje comercial**,
> cada uno con su mensaje. Compruebas los tres requisitos de un segmento útil —accionable, medible y
> estable— retrocediendo el reloj 180 días. Todo con `groupby` y `quantile`: hoy no hace falta ningún
> algoritmo. `KMeans` está en el apéndice, como contraste y fuera de los noventa minutos.
>
> **Entrega** · Proyecto de medio semestre. Este cuaderno ejecutado, la tabla de segmentos con
> clientes, facturación y mensaje comercial, la prueba de estabilidad, y una página de recomendación
> comercial. Se presenta el 7 de octubre.
> Nombre de archivo: `lab_08_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              Path("/content/CursoAnalisisDatos_IA_2026/sitio/datos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Quién es un cliente valioso

La respuesta fácil es «el que más compra», y es la que está mal. Un cliente que compró mucho hace dos
años y no vuelve no vale lo mismo que uno que compra poco cada quince días. Y el que compra mucho con
descuento puede dejar menos margen que uno que compra la mitad a precio de lista.

Por eso el valor de un cliente se mide en cuatro ejes y no en uno:

| eje | qué mide | pregunta de negocio |
|---|---|---|
| **R** · recencia | días desde su última compra | ¿sigue siendo cliente? |
| **F** · frecuencia | número de facturas | ¿es un hábito o fue una casualidad? |
| **M** · monto | facturación acumulada | ¿cuánto mueve? |
| **P** · margen | lo que queda después del costo | ¿cuánto **deja**? |

In [ ]:
ventas = pd.read_csv(DATOS / "ventas_limpias.csv", parse_dates=["fecha"])
clientes = pd.read_csv(DATOS / "clientes.csv", parse_dates=["fecha_alta"])
productos = pd.read_csv(DATOS / "productos.csv")
clientes["ciudad"] = clientes["ciudad"].str.strip().str.title().replace({"Guayaquíl": "Guayaquil"})

v = ventas.merge(productos[["producto_id", "categoria", "costo_unitario"]],
                 on="producto_id", how="left", validate="m:1")
v["monto"] = v["cantidad"] * v["precio_unitario"] * (1 - v["descuento"])
v["margen"] = v["monto"] - v["cantidad"] * v["costo_unitario"]

HOY = v["fecha"].max()
print(f"{len(v):,} líneas · {v['cliente_id'].nunique():,} clientes con movimiento · "
      f"facturación neta {v['monto'].sum():,.2f}")
print(f"fecha de corte del análisis (última fecha de la base): {HOY:%d-%m-%Y}\n")

# El mismo cliente visto por los cuatro ejes: el ranking cambia según cuál mires.
ejes = pd.DataFrame({
    "monto": v.groupby("cliente_id")["monto"].sum(),
    "facturas": v[~v["es_devolucion"]].groupby("cliente_id")["factura_id"].nunique(),
    "ultima_compra": v[~v["es_devolucion"]].groupby("cliente_id")["fecha"].max(),
})
ejes["dias_sin_comprar"] = (HOY - ejes["ultima_compra"]).dt.days
print("Los cinco clientes que más han facturado en la historia de Comercial Andina:")
print(ejes.nlargest(5, "monto")[["monto", "facturas", "dias_sin_comprar"]].round(2).to_string())
print("\nY los cinco que más facturaron entre los que llevan más de un año sin comprar:")
print(ejes.query("dias_sin_comprar > 365").nlargest(5, "monto")[["monto", "facturas", "dias_sin_comprar"]]
      .round(2).to_string())

Los cinco primeros son clientes activos y sanos. Los cinco de abajo facturaron miles de dólares y
llevan más de un año sin aparecer: en una lista ordenada solo por monto estarían arriba, y el equipo
comercial los trataría como si siguieran ahí. **El monto sin la recencia miente.**

## 2. Dos decisiones antes de calcular nada

Hay dos problemas que hay que resolver *antes* de la primera suma, porque después ya están dentro del
número y no se ven. Los dos se deciden por escrito y se cuantifican.

**Problema 1 · las ventas que no se pueden atribuir a nadie.** La tabla limpia del curso ya tiene todos
los `cliente_id`, pero eso es el resultado del trabajo de la semana 4, no el estado natural del
archivo. Así llega el original desde el punto de venta.

In [ ]:
crudo = pd.read_csv(DATOS / "ventas.csv")
crudo["monto"] = crudo["cantidad"] * crudo["precio_unitario"] * (1 - crudo["descuento"])
sin_id = crudo["cliente_id"].isna()

print(f"líneas sin cliente_id : {sin_id.sum():,} ({sin_id.mean():.2%} del archivo original)")
print(f"facturas afectadas    : {crudo.loc[sin_id, 'factura_id'].nunique():,}")
print(f"dinero involucrado    : {crudo.loc[sin_id, 'monto'].sum():,.2f} "
      f"({crudo.loc[sin_id, 'monto'].sum() / crudo['monto'].sum():.2%} de la facturación)")
print(f"\nventas atribuibles a un cliente concreto: "
      f"{1 - crudo.loc[sin_id, 'monto'].sum() / crudo['monto'].sum():.2%}")

**Decisión escrita: las ventas sin `cliente_id` quedan fuera del análisis RFM y se declara la
limitación.** No se pueden repartir entre los clientes conocidos —eso inventaría comportamiento— ni se
pueden agrupar bajo una etiqueta común, porque 5 187 facturas distintas no son un cliente. La frase que
va al informe es: *«el análisis RFM cubre el 92,51 % de la facturación; el 7,49 % restante corresponde
a ventas sin identificación de cliente y no es atribuible»*. Sin esa frase, cualquiera que sume los
segmentos y no llegue al total va a pensar que el análisis está mal.

**Problema 2 · las devoluciones.** En la base hay 1 862 líneas con cantidad negativa y nota de crédito.
Hay tres formas de tratarlas y solo una es correcta.

In [ ]:
compras = v[~v["es_devolucion"]]
devoluciones = v[v["es_devolucion"]]

# A · ignorarlas: sumar solo las ventas
monto_ignorando = compras.groupby("cliente_id")["monto"].sum()
# B · eliminar al cliente que devolvió: absurdo, pero se ve
clientes_con_devolucion = devoluciones["cliente_id"].nunique()
# C · netearlas: restar la devolución del cliente que la hizo
monto_neto = v.groupby("cliente_id")["monto"].sum()

efecto = (monto_ignorando - monto_neto).sort_values(ascending=False)
print(f"A · ignorar las devoluciones : {monto_ignorando.sum():,.2f}  "
      f"({monto_ignorando.sum() / monto_neto.sum() - 1:+.2%} sobre el valor real)")
print(f"C · netearlas (correcto)     : {monto_neto.sum():,.2f}\n")
print(f"clientes que han devuelto algo alguna vez : {clientes_con_devolucion:,} "
      f"({clientes_con_devolucion / v['cliente_id'].nunique():.1%} de la cartera)")
print(f"clientes cuyo valor cae más de un 10 %    : {int(((efecto / monto_ignorando) > 0.10).sum()):,}")
print(f"la devolución más grande de un cliente    : {efecto.max():,.2f} "
      f"({efecto.idxmax()})")

**Decisión escrita: las devoluciones se restan al cliente que las hizo.** Ignorarlas infla la cartera
un 2,42 % y, peor, infla más a unos clientes que a otros: hay 73 clientes cuyo valor real es más de un
10 % inferior al que aparece si solo se suman las ventas. Eliminar al cliente que devolvió —la opción
B, que se ve más de lo que parece— borraría a 901 clientes, el 51,5 % de la cartera, entre ellos los
mejores: el que más compra es también el que más devuelve.

Con las dos decisiones tomadas, ya se puede calcular.

## 3. R, F, M y P por cliente

Una fila por cliente, cuatro columnas y una regla explícita en cada una. La recencia y la frecuencia se
calculan **solo sobre compras** —una devolución no es una visita a la tienda—, mientras que el monto y
el margen usan el neto. Esa asimetría es deliberada y hay que poder defenderla.

In [ ]:
rfmp = pd.DataFrame({
    "recencia": (HOY - compras.groupby("cliente_id")["fecha"].max()).dt.days,
    "frecuencia": compras.groupby("cliente_id")["factura_id"].nunique(),
    "monto": v.groupby("cliente_id")["monto"].sum(),
    "margen": v.groupby("cliente_id")["margen"].sum(),
    "primera_compra": compras.groupby("cliente_id")["fecha"].min(),
}).dropna()
rfmp["antiguedad"] = (HOY - rfmp["primera_compra"]).dt.days
rfmp["tasa_margen"] = rfmp["margen"] / rfmp["monto"] * 100
rfmp["ticket_medio"] = rfmp["monto"] / rfmp["frecuencia"]

print(f"clientes con RFM calculado : {len(rfmp):,} de los {len(clientes):,} del padrón")
print(f"clientes sin ninguna compra: {len(clientes) - len(rfmp):,}  ← existen, valen 0 y hay que decirlo")
print(f"cifra de control · suma del monto por cliente = {rfmp['monto'].sum():,.2f} "
      f"(facturación neta total: {v['monto'].sum():,.2f})\n")
rfmp[["recencia", "frecuencia", "monto", "margen"]].describe().round(2)

In [ ]:
# ✅ Comprobación 1 · la tabla RFM+P tiene que cuadrar contra la cifra de control
assert len(rfmp) == 1751, f"Deberías tener 1 751 clientes con RFM y tienes {len(rfmp)}"
assert abs(rfmp["monto"].sum() - 2806650.55) < 1.0, \
    "Tu facturación por cliente no cuadra con la cifra de control 2 806 650,55: revisa si neteaste las devoluciones"
assert rfmp["recencia"].min() >= 0, "Hay recencias negativas: alguna fecha quedó por delante de HOY"
assert rfmp["monto"].isna().sum() == 0, "Hay clientes con monto vacío: revisa el dropna() de la tabla"

print("Comprobación 1 superada ✓  1 751 clientes y 2 806 650,55 de facturación atribuida")

📌 El monto tiene una media de 1 602,88 y una mediana de 173,11: una razón de 9,26. El cliente medio no
existe, otra vez. Por eso **no se corta por la media ni por umbrales redondos**, sino por quintiles: el
20 % que más compra, el siguiente 20 %, y así. Los quintiles se adaptan a la forma de la distribución
en lugar de imponerle una.

## 4. Del valor al puntaje: quintiles con `qcut`

`qcut` parte una variable en cinco grupos con el mismo número de clientes cada uno. Ojo con la
recencia: **menos días es mejor**, así que sus etiquetas van al revés.

In [ ]:
# R invertida: menos días sin comprar = mejor puntaje.
rfmp["R"] = pd.qcut(rfmp["recencia"], 5, labels=[5, 4, 3, 2, 1]).astype(int)

# F, M y P tienen muchos empates (cientos de clientes con 3 facturas exactas). Sin rank(method="first")
# qcut falla con "Bin edges must be unique" o deja quintiles de tamaños muy distintos.
for col, letra in [("frecuencia", "F"), ("monto", "M"), ("margen", "P")]:
    rfmp[letra] = pd.qcut(rfmp[col].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

cortes = pd.DataFrame({
    "R · recencia (días)": rfmp.groupby("R")["recencia"].agg(lambda s: f"{s.min():.0f} – {s.max():.0f}"),
    "F · facturas": rfmp.groupby("F")["frecuencia"].agg(lambda s: f"{s.min():.0f} – {s.max():.0f}"),
    "M · monto": rfmp.groupby("M")["monto"].agg(lambda s: f"{s.min():,.0f} – {s.max():,.0f}"),
    "P · margen": rfmp.groupby("P")["margen"].agg(lambda s: f"{s.min():,.0f} – {s.max():,.0f}"),
})
cortes.index.name = "puntaje"
print("A qué corresponde cada puntaje (1 = peor quintil, 5 = mejor):\n")
print(cortes.to_string())
print(f"\nclientes por quintil: {rfmp['M'].value_counts().sort_index().tolist()}")

Los cortes salen de los datos y no de la intuición: el quintil 5 de monto empieza en 1 436 dólares y
el quintil 1 termina en 63. Si alguien hubiera propuesto «cliente grande es el que pasa de 1 000», se
habría llevado por delante a parte del quintil 4 y habría dejado dentro a nadie del 5. Los umbrales
redondos son siempre una opinión disfrazada de criterio.

Con los cuatro puntajes ya se puede escribir la ficha de cualquier cliente en cuatro dígitos.

In [ ]:
rfmp["puntaje"] = (rfmp["R"].astype(str) + rfmp["F"].astype(str)
                   + rfmp["M"].astype(str) + rfmp["P"].astype(str))
rfmp["suma_rfm"] = rfmp[["R", "F", "M"]].sum(axis=1)

celdas = (rfmp.groupby("puntaje")
          .agg(clientes=("monto", "size"), monto=("monto", "sum"))
          .assign(pct=lambda d: d["monto"] / d["monto"].sum() * 100)
          .sort_values("monto", ascending=False))

print(f"combinaciones posibles: 5^4 = {5 ** 4}   ·   combinaciones que existen: {len(celdas)}\n")
print("Las seis celdas que más facturan:")
print(celdas.head(6).round(2).to_string())
print(f"\nLa celda 5555 —lo mejor en las cuatro— tiene {celdas.loc['5555', 'clientes']:.0f} clientes "
      f"y el {celdas.loc['5555', 'pct']:.2f} % de la facturación.")

## 5. La matriz RFM

Cuatro dígitos por cliente son precisos e ilegibles. La matriz RFM colapsa el puntaje a dos ejes —el
tiempo y el hábito— y deja ver dónde está la cartera y dónde está el dinero, que no es el mismo sitio.

In [ ]:
conteo = pd.crosstab(rfmp["R"], rfmp["F"])
dinero = pd.crosstab(rfmp["R"], rfmp["F"], values=rfmp["monto"], aggfunc="sum").fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
sns.heatmap(conteo, annot=True, fmt="d", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set_title("Cuántos clientes hay en cada celda", fontsize=11)
sns.heatmap(dinero / 1000, annot=True, fmt=",.0f", cmap="Greens", ax=axes[1], cbar=False)
axes[1].set_title("Cuánto factura cada celda (miles de dólares)", fontsize=11)
for ax in axes:
    ax.set_xlabel("F · frecuencia (1 = compra menos)")
    ax.set_ylabel("R · recencia (1 = hace más que no compra)")
    ax.invert_yaxis()
fig.suptitle("La cartera se reparte casi uniforme; el dinero está todo en una esquina",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

esquina = dinero.loc[5, 5]
print(f"celda R5-F5 : {conteo.loc[5, 5]} clientes ({conteo.loc[5, 5] / len(rfmp):.1%} de la cartera) "
      f"y {esquina:,.2f} ({esquina / rfmp['monto'].sum():.1%} de la facturación)")
print(f"celda R1-F1 : {conteo.loc[1, 1]} clientes ({conteo.loc[1, 1] / len(rfmp):.1%}) "
      f"y {dinero.loc[1, 1]:,.2f} ({dinero.loc[1, 1] / rfmp['monto'].sum():.1%})")

📌 **155 clientes en la esquina R5-F5 —el 8,9 % de la cartera— facturan el 34,0 % del negocio.** En la
esquina opuesta, 169 clientes —casi los mismos— facturan el 0,5 %. Los dos grupos tienen tamaño
parecido y valen cien veces distinto. Ese contraste es todo el argumento de la segmentación: **tratar
igual a los dos es tirar dinero en un lado y perderlo en el otro.**

## 6. La P: el margen que casi nadie calcula

La P se calcula igual que la M pero restando el costo del producto. Su promesa es encontrar al cliente
que compra mucho y deja poco, ese que aparece en el podio de ventas y no debería. Aquí la calculamos
y, sobre todo, comprobamos si aporta algo. Porque a veces no aporta, y eso también hay que reportarlo.

In [ ]:
r_mp = rfmp["monto"].corr(rfmp["margen"])
discrepantes = ((rfmp["M"] >= 4) & (rfmp["P"] <= 2)).sum()

print(f"correlación entre monto y margen : {r_mp:.4f}")
print(f"clientes con M alto (≥4) y P bajo (≤2) : {discrepantes}")
p5, p95 = rfmp["tasa_margen"].quantile([0.05, 0.95])
print(f"tasa de margen por cliente · mediana {rfmp['tasa_margen'].median():.2f} % · "
      f"desviación {rfmp['tasa_margen'].std():.2f} pp")
print(f"  del percentil 5 al 95 va de {p5:.2f} % a {p95:.2f} %: cinco puntos de recorrido\n")

# ¿De dónde sale la (poca) variación que hay? De la mezcla de categorías que compra cada cliente.
mezcla = v.pivot_table(index="cliente_id", columns="categoria", values="monto", aggfunc="sum").fillna(0)
mezcla = mezcla.div(mezcla.sum(axis=1), axis=0) * 100
rfmp = rfmp.join(mezcla.add_prefix("% "))

print("Correlación de la tasa de margen del cliente con lo que compra:")
for cat in mezcla.columns:
    print(f"  % de {cat:17s} {rfmp['tasa_margen'].corr(rfmp['% ' + cat]):+.3f}")

vip_potencial = rfmp[rfmp["M"] == 5]
recorrido = vip_potencial["tasa_margen"].max() - vip_potencial["tasa_margen"].min()
print(f"\nEntre los {len(vip_potencial)} clientes del quintil 5 de monto, la tasa de margen va de "
      f"{vip_potencial['tasa_margen'].min():.2f} % a {vip_potencial['tasa_margen'].max():.2f} % "
      f"({recorrido:.2f} puntos de recorrido).")
print(f"Sobre el cliente más grande ({vip_potencial['monto'].max():,.2f} acumulados) esos "
      f"{recorrido:.2f} puntos valen {vip_potencial['monto'].max() * recorrido / 100:,.2f}.")

⚠️ **Aquí la P no discrimina, y hay que decirlo con el número.** La correlación entre monto y margen es
0,9999 y **ni un solo cliente** tiene monto alto con margen bajo: en Comercial Andina todo el mundo
compra al mismo precio de lista y con descuentos parecidos, así que la tasa de margen apenas se mueve
entre el 41,8 % y el 46,7 % del percentil 5 al 95.

Eso **no** significa que el ejercicio sobre:

1. Se calculó y se comprobó, en lugar de suponerlo. El diagnóstico *«en esta empresa la P no añade
   información al ranking»* es un hallazgo, y evita construir un cuarto eje decorativo.
2. La poca variación que existe tiene causa medible: la mezcla de categorías. La tasa de margen
   correlaciona +0,53 con el porcentaje que el cliente compra en Bebidas y −0,52 con el de Abarrotes.
   Ahí sí hay una acción: mover la mezcla.
3. En un negocio con precios negociados —cualquier distribuidor real— la P es la que descubre al
   cliente grande que no deja nada. El cálculo es idéntico; lo que cambia es el resultado.

Por eso el segmento se construye con R, F y M, y la P se reporta al lado como diagnóstico.

## 7. Del puntaje al segmento con nombre

Aquí el análisis se convierte en algo que alguien puede usar. Las reglas son explícitas, discutibles y
auditables, que es lo que se le pide a una segmentación comercial. Y el segmento «Nuevo» no sale del
puntaje sino de la antigüedad: castigar por poca frecuencia a quien compró por primera vez hace dos
meses sería absurdo.

In [ ]:
def segmentar(fila):
    if fila["antiguedad"] <= 120:                                   return "Nuevo"
    if fila["R"] >= 4 and fila["F"] >= 4 and fila["M"] >= 4:        return "VIP"
    if fila["R"] >= 3 and fila["F"] >= 3:                           return "Leal"
    if fila["R"] <= 2 and (fila["F"] >= 3 or fila["M"] >= 4):       return "En riesgo"
    if fila["R"] <= 2 and fila["F"] <= 2:                           return "Perdido"
    return "Ocasional"


rfmp["segmento"] = rfmp.apply(segmentar, axis=1)
ORDEN = ["VIP", "Leal", "En riesgo", "Ocasional", "Nuevo", "Perdido"]

resumen = (rfmp.groupby("segmento")
    .agg(clientes=("monto", "size"), facturacion=("monto", "sum"), margen=("margen", "sum"),
         recencia_mediana=("recencia", "median"), facturas_mediana=("frecuencia", "median"),
         monto_mediano=("monto", "median"))
    .reindex(ORDEN))
resumen["% clientes"] = resumen["clientes"] / resumen["clientes"].sum() * 100
resumen["% facturación"] = resumen["facturacion"] / resumen["facturacion"].sum() * 100
resumen["valor medio"] = resumen["facturacion"] / resumen["clientes"]

print(f"clientes segmentados: {resumen['clientes'].sum():,} · "
      f"suma de la facturación: {resumen['facturacion'].sum():,.2f} ← cuadra con la cifra de control\n")
resumen.round(2)

📌 La foto del negocio en seis líneas: **369 VIP, el 21,07 % de la cartera, facturan el 62,28 %.** En
el otro extremo, 423 clientes perdidos —el 24,16 %, uno de cada cuatro— aportan el 0,85 %. Y en medio
está el grupo que decide el año que viene: **276 clientes en riesgo que valen 376 901,79, el 13,43 %
del negocio, y que llevan una mediana de 224 días sin comprar.**

Un segmento sin mensaje no es un segmento: es una etiqueta. El mensaje se escribe en una frase, tiene
un destinatario y se puede ejecutar la semana que viene.

In [ ]:
acciones = pd.DataFrame([
    ("VIP", "Retener y proteger", "Gestor de cuenta asignado y reposición programada. Cero descuento: "
     "no lo necesitan y regalarlo cuesta margen."),
    ("Leal", "Aumentar la cesta", "Recomendación cruzada de las categorías que no compran, empezando "
     "por Bebidas, que deja 53,96 % de margen."),
    ("En riesgo", "Reactivar ya", "Llamada del vendedor, no correo. Es el único segmento donde el "
     "descuento se justifica: hay 376 901,79 en juego."),
    ("Ocasional", "Convertir en hábito", "Segunda compra guiada: recordatorio a los 30 días con el "
     "mismo producto que ya compró."),
    ("Nuevo", "Asegurar la segunda compra", "Bienvenida y seguimiento a 15 y 45 días. Todavía no se "
     "sabe cuánto valen: el objetivo es descubrirlo."),
    ("Perdido", "No invertir, medir", "Una campaña masiva de bajo costo y se cierra el caso. "
     "Recuperarlos cuesta más de lo que valen."),
], columns=["segmento", "objetivo", "mensaje comercial"]).set_index("segmento").reindex(ORDEN)

fig, ax = plt.subplots(figsize=(10.5, 4.2))
x = np.arange(len(ORDEN))
ax.bar(x - 0.2, resumen["% clientes"], 0.4, label="% de clientes", color="#B0B0B0")
ax.bar(x + 0.2, resumen["% facturación"], 0.4, label="% de la facturación", color="#4C72B0")
for i, seg in enumerate(ORDEN):
    ax.text(i + 0.2, resumen.loc[seg, "% facturación"] + 1.2,
            f"{resumen.loc[seg, '% facturación']:.1f} %", ha="center", fontsize=9)
ax.set_xticks(x, ORDEN)
ax.set_ylabel("% del total")
ax.set_title("Uno de cada cuatro clientes está perdido y aporta el 0,85 %; el 21 % VIP aporta el 62 %")
ax.legend()
plt.tight_layout()
plt.show()

print(acciones.to_string())

### Los tres requisitos de un segmento útil

Un segmento sirve si cumple tres cosas, y las tres se comprueban con datos, no con opinión:

- **Accionable** · existe una acción distinta para él. Si dos segmentos reciben el mismo mensaje,
  sobra uno.
- **Medible** · se puede contar cuántos son, cuánto valen y si el número se mueve.
- **Estable** · no cambia de composición cada semana. Si un cliente entra y sale del segmento cada
  mes, la campaña llega tarde siempre.

La estabilidad es la que casi nadie comprueba, y se comprueba retrocediendo el reloj.

In [ ]:
CORTE = HOY - pd.Timedelta(days=180)
pasado = v[v["fecha"] <= CORTE]
compras_pasado = pasado[~pasado["es_devolucion"]]

antes = pd.DataFrame({
    "recencia": (CORTE - compras_pasado.groupby("cliente_id")["fecha"].max()).dt.days,
    "frecuencia": compras_pasado.groupby("cliente_id")["factura_id"].nunique(),
    "monto": pasado.groupby("cliente_id")["monto"].sum(),
    "antiguedad": (CORTE - compras_pasado.groupby("cliente_id")["fecha"].min()).dt.days,
}).dropna()
antes["R"] = pd.qcut(antes["recencia"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
antes["F"] = pd.qcut(antes["frecuencia"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
antes["M"] = pd.qcut(antes["monto"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
antes["segmento"] = antes.apply(segmentar, axis=1)

comunes = antes.index.intersection(rfmp.index)
estable = (antes.loc[comunes, "segmento"] == rfmp.loc[comunes, "segmento"])

print(f"clientes presentes en las dos fotos (hace 180 días y hoy): {len(comunes):,}")
print(f"conservan el mismo segmento : {estable.sum():,} ({estable.mean():.1%})")
print(f"cambian de segmento         : {(~estable).sum():,} ({1 - estable.mean():.1%})\n")

movimiento = pd.crosstab(antes.loc[comunes, "segmento"], rfmp.loc[comunes, "segmento"]).reindex(
    index=ORDEN, columns=ORDEN).fillna(0).astype(int)
print("De qué segmento a qué segmento (filas = hace 180 días, columnas = hoy):")
print(movimiento.to_string())

diagonal = pd.Series({seg: movimiento.loc[seg, seg] / movimiento.loc[seg].sum() * 100
                      for seg in ORDEN if movimiento.loc[seg].sum() > 0})
print("\n% de cada segmento que sigue en el mismo sitio medio año después:")
print(diagonal.round(1).to_string())

print("\nComprobación de accionabilidad: ¿algún par de segmentos comparte objetivo?",
      acciones["objetivo"].duplicated().any())

**El 48,2 % conserva su segmento medio año después**, y el número se reporta tal cual aunque no sea el
que uno querría. Mirado por segmento, la lectura cambia y mejora: los dos extremos aguantan —el VIP
retiene el 65,7 % y el Perdido el 68,1 %— y quienes bailan son los del medio, En riesgo con 38,5 % y
Ocasional con 32,3 %. Del Nuevo no sobrevive ninguno, y eso no es inestabilidad sino aritmética: la
antigüedad solo crece.

Los movimientos tienen dirección y sentido de negocio: **42 clientes que hace medio año eran VIP hoy
están En riesgo** y 88 Perdidos volvieron a comprar y hoy son Ocasionales. La consecuencia operativa
está en ese 48,2 %: esta segmentación **se recalcula cada mes, no cada año**, y las campañas se dirigen
a la foto del mes. Un segmento que se mueve permite medir campañas; lo que no se puede es tratarlo como
una etiqueta permanente.

In [ ]:
# ✅ Comprobación 2 · los tres requisitos, comprobados con números
assert resumen["clientes"].sum() == len(rfmp), \
    "La suma de los segmentos no da el total de clientes: alguna regla de segmentar() no devuelve nada"
assert not acciones["objetivo"].duplicated().any(), \
    "Dos segmentos comparten objetivo comercial: si reciben el mismo mensaje, sobra uno"
assert abs(resumen.loc["VIP", "% facturación"] - 62.28) < 0.5, \
    "Los VIP deberían concentrar el 62,28 % de la facturación: revisa los cortes de qcut"
assert 0.40 < estable.mean() < 0.56, \
    "La estabilidad a 180 días debería rondar el 48 %: revisa la fecha de CORTE"

print("Comprobación 2 superada ✓  accionable (6 objetivos distintos), medible y estable al 48,2 %")

### 🌶️ Ejercicio 1 — Guiado

Añade una columna `tipo_cliente` y `ciudad` a la tabla RFM (vienen de `clientes.csv`) y contesta tres
preguntas con código: ¿qué proporción de cada segmento es mayorista?, ¿hay alguna ciudad con más
clientes perdidos de lo que le tocaría por tamaño?, y ¿el canal de captación predice el segmento? Cada
respuesta, una tabla y una frase.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: rfmp.join(clientes.set_index("cliente_id")[["tipo_cliente", "ciudad", "canal_captacion"]])
# Pista 2: pd.crosstab(..., normalize="index") da el reparto dentro de cada segmento
# Pista 3: para «más de lo que le tocaría», compara el % de perdidos de la ciudad contra el % global.
#          Ojo con las ciudades pequeñas: Loja tiene pocos clientes y el porcentaje se mueve mucho

### 🔥 Desafío

Ponle precio al segmento **En riesgo**. Son 276 clientes que valen 376 901,79 acumulados. Estima:
(a) cuánto facturaban al mes cuando estaban activos, (b) cuánto se pierde al año si no se hace nada,
(c) cuánto costaría una campaña de reactivación con un descuento del 10 % y una llamada por cliente, y
(d) qué tasa de recuperación mínima hace que la campaña valga la pena. Escribe el resultado como una
recomendación de tres líneas para el gerente comercial.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: la facturación mensual cuando estaban activos = monto / (antiguedad - recencia) * 30
# Pista 2: el costo de la llamada supónlo y decláralo (por ejemplo 4 dólares por contacto);
#          lo que se califica es que el supuesto esté escrito, no que sea el correcto
# Pista 3: el punto de equilibrio es la tasa de recuperación que iguala margen recuperado y costo total
# Pista 4: usa el margen (44,09 % de media), no la facturación: el descuento sale del margen

### 🎯 Reto en clase (15 min)

En equipos, y contra reloj. Cambia **una** regla de la función `segmentar` —el umbral de antigüedad
del Nuevo, el corte de la recencia del En riesgo, el mínimo de frecuencia del VIP— y mide qué pasa:
cuántos clientes se mueven, cuánta facturación cambia de segmento y si alguna acción comercial dejaría
de tener sentido. Después defiende ante el equipo de al lado cuál de las dos versiones es mejor. La
única defensa que no vale es «porque me da más VIP».

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: copia segmentar() con otro nombre, cambia un umbral y compara con pd.crosstab
#   rfmp["segmento_v2"] = rfmp.apply(segmentar_v2, axis=1)
#   pd.crosstab(rfmp["segmento"], rfmp["segmento_v2"])
# Reporta las tres cifras: clientes que se mueven, facturación que se mueve, segmentos que se vacían

## La trampa de hoy

⚠️ **Entregar los segmentos como códigos.** Es la trampa más barata de evitar y la que más proyectos
entierra: el análisis está bien hecho, los quintiles son correctos, y el informe llega a comercial con
un grupo que se llama «4231» o «Cluster 3». Nadie hace nada con «4231». Se mide fácil: pon las dos
tablas al lado y cuenta cuántas acciones distintas habilita cada una.

In [ ]:
por_codigo = (rfmp.groupby("puntaje")
              .agg(clientes=("monto", "size"), facturacion=("monto", "sum"))
              .sort_values("clientes", ascending=False))
por_codigo["acción comercial"] = "—"

con_nombre = resumen[["clientes", "facturacion"]].copy()
con_nombre["acción comercial"] = acciones["objetivo"]

print(f"LO QUE LLEGA A COMERCIAL CON EL PUNTAJE SIN TRADUCIR · {len(por_codigo)} códigos distintos\n")
print(por_codigo.head(6).round(2).to_string())
print(f"... y {len(por_codigo) - 6} códigos más, de los cuales "
      f"{int((por_codigo['clientes'] <= 5).sum())} tienen cinco clientes o menos\n")
print("LO QUE LLEGA CON LOS SEGMENTOS NOMBRADOS\n")
print(con_nombre.round(2).to_string())

# El daño en dinero: la regla que alguien improvisa cuando solo tiene códigos.
saco = rfmp[rfmp["R"] <= 2]
reparto = saco.groupby("segmento").agg(clientes=("monto", "size"), facturacion=("monto", "sum"))
print(f"\nRegla improvisada sobre los códigos: «R ≤ 2 es un perdido, campaña masiva barata y a otra cosa».")
print(f"caen en el saco {len(saco):,} clientes con {saco['monto'].sum():,.2f} de facturación:\n")
print(reparto.round(2).to_string())
salvables = reparto.loc["En riesgo", "facturacion"] / reparto["facturacion"].sum()
print(f"\ndel dinero de ese saco, el {salvables:.1%} está en clientes que todavía se pueden salvar")
print(f"acciones distintas que habilita la tabla de códigos : {por_codigo['acción comercial'].nunique() - 1}")
print(f"acciones distintas que habilita la tabla nombrada   : {con_nombre['acción comercial'].nunique()}")

📌 **Cero acciones contra seis.** Los cuatro dígitos generan 111 códigos distintos —54 de ellos con
cinco clientes o menos—, y nadie escribe 111 mensajes comerciales. Sin traducción, la tabla no se usa:
se archiva.

Y no es solo que no habilite nada, es que habilita lo equivocado. Cuando solo hay códigos, alguien
improvisa una regla razonable —«R ≤ 2 es un perdido»— y en ese saco caen 699 clientes: 423 Perdidos que
en toda su vida facturaron 23 968,45 y **276 En riesgo que valen 376 901,79**. El 94 % del dinero del
saco está en los que todavía se pueden salvar, y la campaña barata los da por muertos.

La traducción no es cosmética. **Nombrar un segmento es declarar qué se hace con él**, y hasta que no
lo declaras no sabes si el segmento sirve. Cuando un grupo se resiste a tener nombre —«los que compran
medio, medianamente seguido»— la señal es que no existe como categoría comercial.

La regla del curso, sin excepciones: **ningún entregable sale con un segmento llamado 4231 ni
Cluster n.** Cada grupo lleva nombre en español, tamaño, valor y una frase de qué se le dice.

## Entregable

Proyecto de medio semestre. Sube `lab_08_apellido.ipynb` más una página de recomendación, con:

- Las dos decisiones previas escritas y cuantificadas: qué se hizo con las 6 273 líneas sin
  `cliente_id` (el 7,49 % de la facturación) y con las 1 862 devoluciones (el 2,42 %).
- La tabla RFM+P por cliente, con la cifra de control cuadrada: la suma del monto por cliente tiene que
  dar 2 806 650,55.
- La tabla de segmentos con los seis nombres, número de clientes, facturación, porcentaje del total y
  **una frase de mensaje comercial por segmento**.
- La comprobación de los tres requisitos: acciones distintas por segmento, cifras medibles y la prueba
  de estabilidad a 180 días con su porcentaje.
- El diagnóstico de la P escrito: si en tu empresa no discrimina, dilo con la correlación al lado.
- Opcional, del apéndice: el contraste con `KMeans` y **la justificación escrita del K elegido**. Suma,
  pero no sustituye a la segmentación por reglas.
- La bitácora de prompts con el prompt final. No se acepta código pegado sin entender: en la defensa se
  pregunta por qué se eligieron esos umbrales.

## Para tu equipo

- El análisis RFM+P completo del negocio del caso es la entrega grande del semestre. Se presenta el 7
  de octubre y se cierra durante el receso; empezar el 5 no funciona, porque la parte lenta no es el
  código sino discutir los umbrales con alguien que conozca el negocio.
- Si su empresa no tiene margen por producto, calculen R, F y M y **escriban que la P no se pudo
  calcular y por qué**. Es una limitación declarada, no un hueco.
- La página de recomendación se escribe para el gerente comercial, no para el profesor: un segmento por
  párrafo, con cuántos son, cuánto valen, qué se les dice y cómo se mide si funcionó. Si un párrafo no
  termina en una acción concreta, ese segmento sobra.

## Apéndice · para profundizar fuera de clase

Lo que sigue **no compite por los noventa minutos de clase**: es opcional. El cronograma de la semana
es segmentación por reglas de negocio, y `KMeans` entra como **contraste** —la máquina encontrando los
grupos que el equipo ya definió a mano—, no como el tema central. Se ejecuta después de haber corrido
todas las celdas anteriores.

### A1 · Cómo se distribuyen los cuatro ejes

Los histogramas de recencia, frecuencia, monto y margen. Las tres últimas están tan sesgadas que la media no describe a nadie: por eso el paso siguiente son los quintiles.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (col, titulo) in zip(axes, [
        ("recencia", "R · días sin comprar"), ("frecuencia", "F · facturas"),
        ("monto", "M · facturación"), ("margen", "P · margen")]):
    ax.hist(rfmp[col], bins=45, color="#4C72B0", edgecolor="white")
    ax.axvline(rfmp[col].median(), color="#C44E52", linewidth=2)
    ax.set_title(f"{titulo}\nmediana {rfmp[col].median():,.0f}", fontsize=10)
    ax.set_ylabel("")
axes[0].set_ylabel("clientes")
fig.suptitle("Las cuatro variables están sesgadas a la derecha: la media no sirve para cortar",
             fontsize=12, y=1.06)
plt.tight_layout()
plt.show()

for col in ["recencia", "frecuencia", "monto", "margen"]:
    print(f"{col:11s} media {rfmp[col].mean():>10,.2f} · mediana {rfmp[col].median():>9,.2f} "
          f"· razón {rfmp[col].mean() / rfmp[col].median():.2f} · máximo {rfmp[col].max():>10,.2f}")

### A2 · KMeans como contraste

Ahora al revés: en lugar de decirle a la máquina dónde están los grupos, se los dejamos encontrar.
KMeans agrupa por distancia, así que **la escala manda**: sin escalar, el monto (que llega a 28 343)
aplastaría a la recencia (que llega a 922) y a la frecuencia (que llega a 65).

`RobustScaler` centra en la mediana y divide por el rango intercuartílico, en lugar de usar media y
desviación estándar como `StandardScaler`. Con distribuciones tan sesgadas como estas es la elección
correcta: un puñado de clientes enormes no debería definir la escala de toda la cartera.

In [ ]:
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

VARIABLES = ["recencia", "frecuencia", "monto", "margen"]
X = rfmp[VARIABLES]
X_esc = RobustScaler().fit_transform(X)

busqueda = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_esc)
    busqueda.append((k, km.inertia_, silhouette_score(X_esc, km.labels_)))
busqueda = pd.DataFrame(busqueda, columns=["k", "inercia", "silueta"])
busqueda["caída de inercia %"] = -busqueda["inercia"].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.8))
axes[0].plot(busqueda["k"], busqueda["inercia"], marker="o", color="#4C72B0")
axes[0].axvline(4, color="#C44E52", linestyle="--")
axes[0].set_title("Método del codo: la caída se aplana a partir de 5", fontsize=11)
axes[0].set_xlabel("número de grupos (k)"); axes[0].set_ylabel("inercia")

axes[1].plot(busqueda["k"], busqueda["silueta"], marker="o", color="#55A868")
axes[1].axvline(4, color="#C44E52", linestyle="--")
axes[1].set_title("Silueta: máxima en k = 2 y casi plana hasta k = 8", fontsize=11)
axes[1].set_xlabel("número de grupos (k)"); axes[1].set_ylabel("coeficiente de silueta")
fig.suptitle("Las dos métricas no coinciden, y ninguna de las dos sabe de negocio", fontsize=12, y=1.04)
plt.tight_layout()
plt.show()

print(busqueda.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

tam5 = pd.Series(KMeans(n_clusters=5, random_state=SEED, n_init=10).fit_predict(X_esc)).value_counts()
print(f"\ntamaño de los grupos con k = 5: {sorted(tam5.tolist(), reverse=True)}")

**La elección de K, defendida.** Las dos métricas dicen cosas distintas y hay que resolverlo por
escrito, porque en la defensa se pregunta.

- El **codo** cae un 55,70 %, un 41,87 % y un 42,86 % hasta k = 5, y a partir de ahí se queda en el
  27,52 %. La última caída grande está entre 4 y 5.
- La **silueta** es máxima en k = 2 (0,8440) y baja monótonamente. Pero una silueta de 0,84 en estos
  datos no significa que haya dos grupos naturales: significa que hay dieciséis clientes gigantescos
  tan lejos de todo que cualquier partición que los aísle puntúa alto. **La métrica está midiendo los
  extremos, no la estructura.**
- Entre k = 4 y k = 5, decide el negocio: con 5, el grupo superior se parte en uno de 46 clientes y
  otro de 12, y el equipo comercial les haría exactamente lo mismo. Con 4 no se pierde casi nada de
  silueta —0,8275 contra 0,8440, diecisiete milésimas— y quedan cuatro grupos que sí se pueden atender
  con recursos distintos.

**K = 4.** Esa es la respuesta, y esta es la justificación completa.

In [ ]:
modelo = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(X_esc)
rfmp["cluster"] = modelo.labels_

perfil = (rfmp.groupby("cluster")[VARIABLES].median()
          .assign(clientes=rfmp.groupby("cluster").size(),
                  facturacion=rfmp.groupby("cluster")["monto"].sum())
          .assign(pct_clientes=lambda d: d["clientes"] / d["clientes"].sum() * 100,
                  pct_facturacion=lambda d: d["facturacion"] / d["facturacion"].sum() * 100))
orden_cluster = perfil.sort_values("monto").index.tolist()

print("Perfil de los cuatro grupos que encontró el algoritmo (valores medianos):\n")
print(perfil.loc[orden_cluster].round(2).to_string())
print(f"\nsilueta con k = 4 : {silhouette_score(X_esc, modelo.labels_):.4f}")

grande = perfil["clientes"].idxmax()
sub = rfmp[rfmp["cluster"] == grande]
print(f"\nEl grupo más grande es el {grande}: {len(sub):,} clientes ({len(sub) / len(rfmp):.1%}).")
print(f"  su recencia va de {sub['recencia'].min():.0f} a {sub['recencia'].max():.0f} días")
print(f"  su frecuencia va de {sub['frecuencia'].min():.0f} a {sub['frecuencia'].max():.0f} facturas")

### A3 · Reglas contra algoritmo: la comparación honesta

El algoritmo encontró cuatro grupos ordenados por tamaño de cliente: la masa, los medianos, los
grandes y los gigantes. Es una escalera de dinero. Y el grupo mayor se lleva 1 441 clientes —el
82,3 %— con recencias que van de 19 a 922 días metidas en la misma caja.

Las dos segmentaciones salen de las mismas cuatro variables y de los mismos 1 751 clientes. La tabla
cruzada dice dónde coinciden y dónde no, y una medida sencilla —cuánta varianza de cada variable
explica cada partición— dice qué encontró cada una.

In [ ]:
cruce = pd.crosstab(rfmp["segmento"], rfmp["cluster"]).reindex(ORDEN)
cruce["total"] = cruce.sum(axis=1)
print("Reglas RFM (filas) contra KMeans (columnas):\n")
print(cruce.to_string())


def varianza_explicada(etiquetas, columna):
    total = rfmp[columna].var()
    dentro = rfmp.groupby(etiquetas)[columna].transform(lambda s: s - s.mean()).var()
    return 1 - dentro / total


comparacion = pd.DataFrame({
    "reglas RFM": [varianza_explicada(rfmp["segmento"], c) for c in VARIABLES],
    "KMeans (k=4)": [varianza_explicada(rfmp["cluster"], c) for c in VARIABLES],
}, index=VARIABLES) * 100

print("\n% de la varianza de cada eje que explica cada segmentación:\n")
print(comparacion.round(1).to_string())

📌 **El algoritmo encontró el eje del dinero; las reglas encontraron el eje del tiempo.** KMeans
explica el 93,6 % de la varianza del monto y solo el 5,0 % de la de la recencia. Las reglas RFM van al
revés: 62,4 % de la recencia y 22,9 % del monto.

Y eso decide la comparación, porque **la recencia es la variable que habilita una acción**. En el grupo
más numeroso de KMeans conviven 423 clientes perdidos y 197 VIP; no existe un mensaje comercial que
sirva para los dos. Los segmentos por reglas se pueden atender por separado desde mañana.

Lo que gana el algoritmo, y no es poco:

- **No necesita que alguien invente los umbrales.** Las reglas de la sección 7 son mías y son
  discutibles; los grupos de KMeans salen de los datos.
- **Encuentra la escalera de tamaño mejor que los quintiles**, porque no está obligado a poner el mismo
  número de clientes en cada grupo: aisló 16 clientes gigantes que los quintiles diluyen dentro de 350.
- **Escala a más variables.** Con cuatro ejes las reglas todavía se escriben a mano; con quince, no.

Lo que pierde:

- **No sabe qué es la recencia.** Trata las cuatro variables como si valieran lo mismo, y en negocio no
  valen lo mismo.
- **Los grupos no tienen nombre**, y ese es el problema de la sección siguiente.
- **No es estable ante el escalado ni la semilla.** Cambia `RobustScaler` por `StandardScaler` y los
  grupos cambian; el negocio, no.

**La conclusión práctica y honesta: se usan los dos.** Las reglas RFM para la acción comercial, porque
son interpretables, estables y discutibles con el gerente. KMeans para descubrir estructura que las
reglas no vieron —esos 16 clientes gigantes son una categoría propia y merecen un trato propio— y para
auditar los umbrales que pusiste a mano.